In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time

links = []
reqions = 4

print("Collecting house links...")

for i in tqdm(range(1,50,1), desc='pages', unit='page'):
    r = requests.get(
        f"https://iranfile.ir/properties/buy-apartment?page={i}&regions={reqions},-14,-1"
    )
    time.sleep(1)
    content = r.content.decode("utf-8")
    soup = BeautifulSoup(content, features='html.parser')

    for a in soup.findAll('a', href=True, attrs={'class': 'grd_search_links'}):
        links.append(a['href'])

    links = list(set(links))

file_path = "HomeDatasetTehran_R4_1404_02.csv"

columns = [
    "h_type", "date", "adress", "loc", "num_floor", "unit_per_floor", "price", "age", 
    "view", "doc_status", "north", "south", "west", "east", "floor", 
    "area", "num_sleep", "telephone", "kitchen", "service", "floor_covering", "open", 
    "parking", "warehouse", "balcony", "equipment"]

df = pd.DataFrame(columns=columns)
equipment_list = [None] * len(links)

print('Extracting house data...')

for index, item in enumerate(tqdm(links, desc='Houses', unit='House')):
    try:
        r = requests.get(item)
        time.sleep(1)
        content = r.content.decode('utf-8')
        soup = BeautifulSoup(content, features='html.parser')
        
        # استخراج مقادیر اولیه
        first_values = []
        for a in soup.findAll(['div','span'], attrs={'class': 'file-data'}):
            first_values.append(a.text.strip())
        
        # پر کردن house_data
        house_data = {col: None for col in columns}
        cols = ['h_type','date','adress','num_floor','unit_per_floor',
               'price','age','view','doc_status']
        num = [2,0,5,6,7,8,10,11,12,13]
        
        for col_index, col in enumerate(cols):
            if (col_index < len(num)) and (num[col_index] < len(first_values)):
                house_data[col] = first_values[num[col_index]]
        
        house_data['loc'] = reqions
        
        # جهت‌ها
        direction = soup.findAll('div', attrs={'class': 'file-active'})
        direction_dict = {"north": 0, "south": 0, "west": 0, "east": 0}
        dir_fa = ["شمالی", "جنوبی", "غربی", "شرقی"]
        dir_en = ["north", "south", "west", "east"]
        
        for d in direction:
            dir_text = d.text.strip()
            if dir_text in dir_fa:
                idx = dir_fa.index(dir_text)
                direction_dict[dir_en[idx]] = 1
        
        house_data.update(direction_dict)
        
        # اطلاعات اضافی
        rows = soup.findAll("td")
        target_columns = ['floor','area','num_sleep','telephone',
                        'kitchen','service','floor_covering']
        desired_values = []
        
        for i in range(1, 14, 2):
            if i < len(rows):
                desired_values.append(rows[i].text.strip())
            else:
                desired_values.append(None)
        
        row_dict = dict(zip(target_columns, desired_values))
        
        # وضعیت ویژگی‌ها
        feature_columns = ['open','parking','warehouse','balcony']
        feature_status = []
        
        for i in [15, 17, 19, 21]:
            if i < len(rows):
                is_checked = 1 if rows[i].find('i', class_='file-checked') else 0
                feature_status.append(is_checked)
            else:
                feature_status.append(0)
        
        row_dict.update(dict(zip(feature_columns, feature_status)))
        house_data.update(row_dict)
        
        # تجهیزات
        e = []
        for eq_item in soup.findAll('div', class_='check-box-info'):
            e.append(eq_item.text.strip())
        
        equipment_list[index] = '_'.join(e) if e else None

        doc_type = None
        for info in soup.findAll("div", class_="info-melk-file-details"):
            if info.find("div") and "وضعیت سند" in info.find("div").text:
                doc_type = info.find("div", class_="file-data").text.strip() if info.find(
                    "div", class_="file-data") else None
                break
        
        house_data['Document_type'] = doc_type  

        pro_status = None
        for info in soup.findAll("div", class_="info-melk-file-details"):
            if info.find("div") and "وضعیت ملک" in info.find("div").text:
                pro_status = info.find("div", class_="file-data").text.strip() if info.find(
                    "div", class_="file-data") else None
                break
        
        house_data['Propertyـstatus'] = pro_status 
        
        # اضافه کردن به DataFrame
        df = pd.concat([df, pd.DataFrame([house_data])], ignore_index=True)
    
    except Exception as e:
        print(f"Error processing {item}: {e}")
        equipment_list[index] = None
        continue

# اضافه کردن لیست تجهیزات به DataFrame
df['equipment'] = equipment_list
df.to_csv(file_path, index=False, sep='\t', encoding='utf-8-sig')